<a href="https://colab.research.google.com/github/taeyoung0524/LoRA-VLM/blob/main/1_5_VLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.5주차 실습: SmolVLM Full Fine-tuning 직접 구현과 Trainer 비교

## 주차 목표

COCO image-caption subset으로 `HuggingFaceTB/SmolVLM-256M-Instruct`를 full fine-tuning한다. 먼저 Hugging Face `Trainer` 없이 PyTorch `DataLoader`, `AdamW`, autograd, scheduler를 직접 사용해 학습 loop를 구현하고, 뒤에서 같은 split과 prompt로 `Trainer` 기반 학습 코드를 비교한다.

## Section 구성

- **Section 1:** 공통 COCO split, zero-shot baseline, shared evaluation helper를 준비한다.
- **Section 2:** `RUN_PYTORCH_FULL_FINETUNING=True`일 때 순수 PyTorch loop로 full fine-tuning을 실행하고 최종 model을 저장한다.
- **Section 3:** `RUN_TRAINER_FULL_FINETUNING=True`일 때 같은 split과 prompt를 사용하는 Trainer fine-tuning 코드를 실행한다.
- **Section 4:** 학습이 완료된 fine-tuning 결과의 caption과 metric을 비교한다.

## 실행 결과

- **저장 경로:** `model/1-5-coco-full-finetuning/pytorch/`, `model/1-5-coco-full-finetuning/trainer/`
- **핵심 산출물:** 최종 fine-tuned model, `train_eval_metrics.json`, `before_after.json`, `method_comparison.json`.


## Colab 환경 설정

Colab에서 실행할 때 Google Drive를 mount하고 프로젝트 경로를 import path에 추가한다.

- **프로젝트 경로:** `/content/drive/MyDrive/VLM-Lecture2`
- **수정 지점:** Drive 폴더명이 다르면 `DRIVE_PROJECT`만 바꾼다.


In [ ]:
# Colab 환경 설정
# Drive 프로젝트를 Python import path에 추가해 Colab에서도 저장소의 utils 모듈을 바로 불러오게 합니다.
# Colab이 아닌 환경에서는 google.colab 모듈이 없으므로 mount 로직을 건너뜁니다.
import importlib.util
import sys

DRIVE_PROJECT = '/content/drive/MyDrive/VLM-Lecture2'  # 본인 Drive 폴더명에 맞게 수정

if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive

    drive.mount('/content/drive')
    if DRIVE_PROJECT not in sys.path:
        sys.path.insert(0, DRIVE_PROJECT)

# 필요할 때만 아래 두 줄의 주석을 해제하세요.
# - Drive 원본을 /content 로컬 디스크로 복사해 I/O 병목을 줄이고 싶을 때
# - 새 Colab 런타임에서 작업 루트와 import path를 한 번에 맞출 때
# gd_mount: Colab Drive 프로젝트를 로컬 작업 경로로 복사하고 import path를 맞춥니다.
# from utils.gd_mount import setup_colab_workdir
# setup_colab_workdir(drive_project=DRIVE_PROJECT, mount_drive=False)


## 로컬 환경 설정

로컬 Jupyter나 VS Code에서 실행할 때 현재 경로의 부모를 탐색해 저장소 루트를 찾는다.

- **탐색 기준:** `index.md`가 있는 디렉터리를 프로젝트 루트로 사용한다.
- **환경 기준:** 로컬에서는 `uv sync` 또는 이미 준비된 `.venv`를 사용한다.


In [ ]:
# 로컬 환경 설정
# 로컬 실행에서는 현재 폴더와 부모 폴더를 탐색해 index.md가 있는 저장소 루트를 찾습니다.
# 작업 디렉터리와 sys.path를 같은 루트로 맞춰 data/, model/, utils/ 상대경로가 일관되게 동작하게 합니다.
import os
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'index.md').exists():
        LOCAL_PROJECT_PATH = candidate
        break
else:
    LOCAL_PROJECT_PATH = Path.cwd()

os.chdir(LOCAL_PROJECT_PATH)
local_project_path = str(LOCAL_PROJECT_PATH)
if local_project_path not in sys.path:
    sys.path.insert(0, local_project_path)
print(f'Local project root: {LOCAL_PROJECT_PATH}')


## GPU 확인

**현재 런타임의 GPU와 PyTorch CUDA 인식 상태를 확인한다.** Full fine-tuning은 GPU 실행을 기본으로 한다.


In [ ]:
# GPU 확인
# Full fine-tuning은 GPU memory를 많이 사용하므로 nvidia-smi와 torch CUDA 인식을 모두 확인합니다.
# nvidia-smi가 없어도 셀은 계속 진행하지만, torch.cuda.is_available()이 False이면 후속 학습은 CPU/오류 경로로 이어집니다.
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi 명령을 찾을 수 없습니다. GPU 런타임인지 확인하세요.')

import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 필요 라이브러리 설치

SmolVLM full fine-tuning 비교에 필요한 패키지를 설치한다. PyTorch 학습 loop는 `Trainer`를 쓰지 않지만, SmolVLM model/processor와 후반부 Trainer 비교를 위해 `transformers`와 `accelerate`가 필요하다.

- **설치 패키지:** `transformers`, `datasets`, `evaluate`, `nltk`, `matplotlib`, `accelerate`, `rich`, `tqdm`, `Pillow`


In [ ]:
# 필요 라이브러리 설치
!pip install -q transformers==4.57.6 datasets==4.8.4 evaluate==0.4.6 nltk==3.9.4 matplotlib==3.10.8 accelerate==1.13.0 rich==13.9.4 tqdm==4.67.3 Pillow==11.3.0
# Colab 기본 설치 패키지는 보존: torch/torchvision/torchaudio


---

# Section 1: 공통 COCO Split과 Zero-shot Baseline

공통 COCO subset, prompt, zero-shot caption을 준비해 PyTorch loop와 Trainer loop가 같은 조건에서 학습되고 평가되게 한다.


## 공통 import

공통 import는 PyTorch 직접 학습 loop, Trainer 비교, caption metric, 최종 모델 저장에 필요한 모듈을 한곳에 모은다.


In [ ]:
# 공통 import

# 이 셀은 직접 PyTorch loop, Hugging Face Trainer 비교, caption 평가, JSON 저장, 시각화에 필요한 import를 한곳에 모읍니다.
# 저장소 utils import는 반복되는 COCO 준비, SmolVLM batch 구성, report 생성 코드를 노트북 밖 공용 함수로 분리한 부분입니다.
import math
import time
import transformers
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# coco_dataloader: COCO image-caption subset 샘플링과 Dataset 생성을 담당합니다.
# coco_dataloader: COCO split과 Dataset 구성을 공용 helper에 맡깁니다.
from utils.coco_dataloader import create_coco_dataset, prepare_coco_caption_dataset_bundle
# device_utils: device 선택, 환경 정보 수집, CUDA 메모리 정리를 담당합니다.
# device_utils: CUDA/CPU device 선택과 memory 정리를 공용 helper로 처리합니다.
from utils.device_utils import get_device_info, release_cuda_memory, resolve_device
# logger_utils: notebook 실행 로그를 일관된 형식으로 남기는 logger를 제공합니다.
# logger_utils: 실행 로그를 notebook 전체에서 같은 형식으로 남깁니다.
from utils.logger_utils import get_logger
# report_utils: full fine-tuning 평가 artifact와 비교 report 구성을 담당합니다.
# report_utils: 학습 결과 report와 method comparison 구조를 만듭니다.
from utils.report_utils import (
    build_full_finetuning_method_comparison_report,
    build_full_finetuning_visualization_sets,
    evaluate_full_finetuning_caption_run,
)
# smolvlm_utils: SmolVLM 로드, generation, SFT collator, dtype helper를 제공합니다.
# smolvlm_utils: SmolVLM 로드, collator, generation 관련 반복 코드를 재사용합니다.
from utils.smolvlm_utils import (
    build_sft_collate_fn,
    generate_caption_splits,
    load_model_and_processor,
    move_batch_to_device,
    resolve_torch_dtype,
)
# training_utils: Trainer 인자 구성, parameter 수 집계, JSON 저장 helper를 제공합니다.
# training_utils: config 직렬화, TrainingArguments 구성, JSON 저장을 공용화합니다.
from utils.training_utils import build_config_payload, build_training_arguments_kwargs, count_parameters, save_json
# visualization: caption 비교 카드와 metric 요약 시각화를 제공합니다.
# visualization: caption 카드와 metric 그래프를 notebook 안에 표시합니다.
from utils.visualization import show_caption_comparison_cards, show_caption_metric_comparison


## 실행 설정

**핵심:** 두 fine-tuning 방식이 공유할 데이터 크기, 하이퍼파라미터, 저장 경로를 정의한다.

- `data_dir`: 공통 split manifest 저장 경로다.
- `output_dir`: 두 방식의 결과를 담는 root 경로다.
- `pytorch_output_dir`: 직접 PyTorch loop 모델과 report 경로다.
- `trainer_output_dir`: Trainer 모델과 report 경로다.


In [ ]:
# 실행 설정

# 두 실행 flag로 시간이 오래 걸리는 PyTorch full FT와 Trainer 비교를 독립적으로 켜고 끌 수 있습니다.
# FinetuneConfig는 데이터 split 크기, 학습 hyperparameter, decoding 길이, artifact 경로를 하나의 실행 계약으로 묶습니다.
# PyTorch loop와 Trainer는 같은 config, 같은 COCO split, 같은 prompt를 사용해야 method comparison이 의미를 갖습니다.
RUN_PYTORCH_FULL_FINETUNING = True
RUN_TRAINER_FULL_FINETUNING = True

FINETUNE_LOGGER = get_logger(__name__)

@dataclass(slots=True)
class FinetuneConfig:
    model_name: str = "HuggingFaceTB/SmolVLM-256M-Instruct"
    prompt: str = "Describe this image in one concise English sentence."
    train_images: int = 1000
    val_images: int = 20
    test_images: int = 50
    compare_images: int = 4
    seed: int = 42
    train_split: str = "train"
    eval_split: str = "validation"
    device: str | None = "cuda"
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 16
    num_train_epochs: int = 1
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    logging_steps: int = 5
    max_new_tokens: int = 48
    run_pytorch_training: bool = RUN_PYTORCH_FULL_FINETUNING
    run_trainer_comparison: bool = RUN_TRAINER_FULL_FINETUNING
    data_dir: Path = Path("data/1-5-coco-full-finetuning")
    output_dir: Path = Path("model/1-5-coco-full-finetuning")
    pytorch_output_dir: Path = Path("model/1-5-coco-full-finetuning/pytorch")
    trainer_output_dir: Path = Path("model/1-5-coco-full-finetuning/trainer")


## Device 및 경로 준비

**핵심:** 저장 경로를 만들고 GPU device 정보를 공통 output root에 저장한다.


In [ ]:
# Device 및 경로 준비

# config를 실제 실행 객체로 만들고, 전체 image 수와 출력 경로를 먼저 확정합니다.
# device_info.json은 어떤 GPU/CPU 조건에서 결과가 생성됐는지 남기는 재현 artifact입니다.
config_ft = FinetuneConfig()
total_images = config_ft.train_images + config_ft.val_images + config_ft.test_images

FINETUNE_LOGGER.info(
    "Starting 1.5 fine-tuning run | model=%s train=%s val=%s test=%s epochs=%s output_dir=%s",
    config_ft.model_name,
    config_ft.train_images,
    config_ft.val_images,
    config_ft.test_images,
    config_ft.num_train_epochs,
    config_ft.output_dir,
)

for output_path in [config_ft.data_dir, config_ft.output_dir, config_ft.pytorch_output_dir, config_ft.trainer_output_dir]:
    output_path.mkdir(parents=True, exist_ok=True)

device = resolve_device(config_ft.device)
device_info = get_device_info(config_ft.device)
device_info_payload = {**device_info, "device": str(device_info["device"])}
save_json(config_ft.output_dir / "device_info.json", device_info_payload)
FINETUNE_LOGGER.info("Using device=%s total_images=%s", device, total_images)


## 데이터 준비

**핵심:** train split과 validation/test split을 한 번만 샘플링해 PyTorch loop와 Trainer loop가 같은 sample 순서를 사용하게 한다.


In [ ]:
# 데이터 준비

# 공용 COCO helper가 train/validation/test/comparison split을 만들고 manifest에 필요한 image id 목록을 반환합니다.
# train_dataset과 val_dataset은 SFT collator가 image-caption row를 batch tensor로 바꾸기 위한 입력입니다.
# caption_splits는 zero-shot, PyTorch fine-tuned, Trainer fine-tuned 모델이 같은 sample 묶음에 caption을 만들도록 고정합니다.
dataset_bundle = prepare_coco_caption_dataset_bundle(config_ft, logger=FINETUNE_LOGGER)
comparison_samples = dataset_bundle["comparison_samples"]
test_image_samples = dataset_bundle["metric_samples"]
train_dataset = create_coco_dataset(dataset_bundle["train_rows"])
val_dataset = create_coco_dataset(dataset_bundle["val_rows"])
caption_splits = {
    "comparison": comparison_samples,
    "test": test_image_samples,
}
save_json(
    config_ft.data_dir / "split_manifest.json",
    dataset_bundle["manifest"],
)

FINETUNE_LOGGER.info(
    "Prepared split | train=%s val=%s test=%s comparison=%s",
    len(train_dataset),
    len(val_dataset),
    len(test_image_samples),
    len(comparison_samples),
)


## Base model 로드와 zero-shot baseline

**핵심:** 학습 전 SmolVLM caption을 저장해 두 방식의 fine-tuning 결과와 같은 test split에서 비교한다.


In [ ]:
# Base model 로드와 zero-shot baseline

# 학습 전 base SmolVLM을 로드해 zero-shot caption을 먼저 생성합니다.
# 이 zero-shot 결과는 fine-tuning 이후 before/after report와 metric comparison의 기준선입니다.
torch_dtype = resolve_torch_dtype(device=device, torch_dtype="auto")
base_bundle = load_model_and_processor(
    model_id=config_ft.model_name,
    device=device,
    torch_dtype=torch_dtype,
    prefer_flash_attention=False,
)
pytorch_processor = base_bundle["processor"]
pytorch_model = base_bundle["model"]
pytorch_model_torch_dtype = base_bundle["torch_dtype"]

FINETUNE_LOGGER.info("Loaded base model with dtype=%s", pytorch_model_torch_dtype)
FINETUNE_LOGGER.info(
    "Model parameter summary | total=%s trainable=%s",
    count_parameters(pytorch_model),
    count_parameters(pytorch_model, trainable_only=True),
)

zero_shot_outputs = generate_caption_splits(
    label="Zero-shot",
    model=pytorch_model,
    processor=pytorch_processor,
    sample_splits=caption_splits,
    device=device,
    prompt=config_ft.prompt,
    max_new_tokens=config_ft.max_new_tokens,
    batch_size=config_ft.per_device_eval_batch_size,
    logger=FINETUNE_LOGGER,
)


---

# Section 2: PyTorch 기본 Loop Full Fine-tuning

Hugging Face `Trainer` 없이 `DataLoader`, `AdamW`, autograd, scheduler로 SmolVLM 전체 parameter를 학습한다.


## PyTorch 학습 helper

**핵심:** Trainer가 숨기던 optimizer step, gradient accumulation, scheduler, validation 평균 loss 계산을 명시적인 PyTorch 함수로 만든다.


In [ ]:
# PyTorch 학습 helper

# Trainer가 내부에서 처리하는 scheduler, autocast, validation loss 계산을 직접 PyTorch loop용 helper로 명시합니다.
# 각 helper는 학습 loop 본문을 짧게 유지하면서도 warmup/decay, mixed precision, evaluation 평균 계산 방식을 드러냅니다.
def build_linear_warmup_decay_scheduler(
    optimizer: torch.optim.Optimizer,
    *,
    total_steps: int,
    warmup_ratio: float,
) -> torch.optim.lr_scheduler.LambdaLR:
    # warmup step 수는 전체 optimizer update 수와 warmup_ratio로 결정합니다.
    warmup_steps = int(total_steps * warmup_ratio)

    # LambdaLR에 넘길 함수는 현재 step의 learning rate multiplier를 반환합니다.
    def lr_lambda(current_step: int) -> float:
        if warmup_steps > 0 and current_step < warmup_steps:
            return float(current_step + 1) / float(warmup_steps)
        # warmup 이후에는 남은 step 비율만큼 learning rate를 선형으로 줄입니다.
        remaining_steps = max(total_steps - current_step, 0)
        decay_steps = max(total_steps - warmup_steps, 1)
        return float(remaining_steps) / float(decay_steps)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def autocast_context(device: Any, torch_dtype: Any | None) -> Any:
    # autocast는 CUDA에서 fp16/bf16을 사용할 때만 의미가 있습니다.
    use_cuda_autocast = getattr(device, "type", None) == "cuda" and torch_dtype in {torch.float16, torch.bfloat16}
    if not use_cuda_autocast:
        return nullcontext()
    return torch.autocast(device_type="cuda", dtype=torch_dtype)


def evaluate_loss_pytorch(
    *,
    model: Any,
    dataloader: DataLoader,
    device: Any,
    torch_dtype: Any | None,
) -> dict[str, float]:
    # validation에서는 dropout을 끄고 gradient를 계산하지 않습니다.
    model.eval()
    total_loss = 0.0
    total_examples = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="PyTorch validation"):
            # collator가 만든 tensor batch를 model과 같은 device로 옮깁니다.
            batch = move_batch_to_device(batch, device)
            batch_size = int(batch["input_ids"].shape[0])
            with autocast_context(device, torch_dtype):
                outputs = model(**batch)
            # batch 크기가 다를 수 있으므로 sample 수로 가중한 loss 합을 누적합니다.
            total_loss += float(outputs.loss.detach().cpu()) * batch_size
            total_examples += batch_size
    eval_loss = total_loss / max(total_examples, 1)
    return {"eval_loss": eval_loss, "eval_samples": float(total_examples)}


## PyTorch DataLoader 구성

**핵심:** `build_sft_collate_fn`으로 image/caption row를 model input tensor와 assistant-only `labels`로 바꾸고, 직접 `DataLoader`에 연결한다.


In [ ]:
# PyTorch DataLoader 구성

# PyTorch loop 전용 checkpoint 경로, config payload, SFT collator, train/validation DataLoader를 구성합니다.
# train DataLoader는 seed가 고정된 shuffle을 사용하고, validation DataLoader는 metric 비교를 위해 순서를 고정합니다.
# max_train_steps는 batch 수가 아니라 gradient accumulation 이후 실제 optimizer update 횟수입니다.
pytorch_model_dir = config_ft.pytorch_output_dir / "smolvlm-pytorch-full-finetuned"
pytorch_config_payload = build_config_payload(config_ft, extra={"sft_label_mask": "assistant_only_v1", "training_method": "pytorch_loop_v1"})
pytorch_collate_fn = build_sft_collate_fn(
    pytorch_processor,
    image_token_id=getattr(pytorch_model.config, "image_token_id", None),
    prompt=config_ft.prompt,
)

generator = torch.Generator()
generator.manual_seed(config_ft.seed)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=config_ft.per_device_train_batch_size,
    shuffle=True,
    generator=generator,
    collate_fn=pytorch_collate_fn,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=config_ft.per_device_eval_batch_size,
    shuffle=False,
    collate_fn=pytorch_collate_fn,
)

num_update_steps_per_epoch = math.ceil(len(train_dataloader) / config_ft.gradient_accumulation_steps)
max_train_steps = max(1, config_ft.num_train_epochs * num_update_steps_per_epoch)
FINETUNE_LOGGER.info(
    "PyTorch loop steps | train_batches=%s update_steps_per_epoch=%s max_train_steps=%s",
    len(train_dataloader),
    num_update_steps_per_epoch,
    max_train_steps,
)


## PyTorch 학습

**핵심:** 직접 PyTorch loop로 full fine-tuning을 실행한 뒤 PyTorch 방식의 최종 model과 processor를 저장한다.


In [ ]:
# PyTorch 학습

# 이 셀은 Hugging Face Trainer 없이 optimizer, scheduler, autocast, gradient accumulation을 직접 연결해 full fine-tuning을 수행합니다.
# skip flag가 꺼져 있을 때도 후속 셀이 안전하게 동작하도록 결과 변수들을 먼저 None 또는 빈 list로 초기화합니다.
# 학습 완료 후 model과 processor를 save_pretrained 형식으로 저장해 Hugging Face checkpoint처럼 재사용할 수 있게 합니다.
pytorch_training_ran = False
pytorch_log_history: list[dict[str, Any]] = []
pytorch_train_result_metrics: dict[str, Any] | None = None

if config_ft.run_pytorch_training:
    FINETUNE_LOGGER.info("Starting direct PyTorch full fine-tuning")
    # 학습 중 decoder cache는 memory를 늘리고 checkpointing과 충돌할 수 있어 끕니다.
    if getattr(pytorch_model.config, "use_cache", None) is not None:
        pytorch_model.config.use_cache = False
    if getattr(device, "type", None) == "cuda" and hasattr(pytorch_model, "gradient_checkpointing_enable"):
        pytorch_model.gradient_checkpointing_enable()

    # full fine-tuning이므로 adapter가 아니라 전체 model parameter를 optimizer에 넘깁니다.
    optimizer = torch.optim.AdamW(
        pytorch_model.parameters(),
        lr=config_ft.learning_rate,
        weight_decay=config_ft.weight_decay,
    )
    # scheduler는 optimizer update가 일어나는 global_step 기준으로 움직입니다.
    scheduler = build_linear_warmup_decay_scheduler(
        optimizer,
        total_steps=max_train_steps,
        warmup_ratio=config_ft.warmup_ratio,
    )
    # fp16 CUDA에서는 GradScaler를 켜고, bf16/fp32에서는 자동으로 꺼집니다.
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=getattr(device, "type", None) == "cuda" and pytorch_model_torch_dtype is torch.float16,
    )

    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    train_start_time = time.perf_counter()
    last_loss = None

    # epoch마다 train mode를 다시 설정해 validation 이후 상태가 섞이지 않게 합니다.
    for epoch_index in range(config_ft.num_train_epochs):
        pytorch_model.train()
        progress = tqdm(train_dataloader, desc=f"PyTorch train epoch {epoch_index + 1}")
        for batch_index, batch in enumerate(progress):
            # image/text tensor batch를 GPU로 옮긴 뒤 forward loss를 계산합니다.
            batch = move_batch_to_device(batch, device)
            with autocast_context(device, pytorch_model_torch_dtype):
                outputs = pytorch_model(**batch)
                loss = outputs.loss
                # gradient accumulation을 쓰므로 loss를 나눠 유효 batch 크기에 맞춥니다.
                scaled_loss = loss / config_ft.gradient_accumulation_steps

            if scaler.is_enabled():
                scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()

            # accumulation 간격이 찼거나 마지막 batch이면 실제 optimizer step을 수행합니다.
            should_step = (
                (batch_index + 1) % config_ft.gradient_accumulation_steps == 0
                or (batch_index + 1) == len(train_dataloader)
            )
            if should_step:
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                last_loss = float(loss.detach().cpu())
                current_lr = float(scheduler.get_last_lr()[0])
                # 직접 loop의 loss/lr 기록을 Trainer log_history와 비슷한 dict 형태로 남깁니다.
                log_record = {
                    "step": global_step,
                    "epoch": epoch_index + 1,
                    "loss": last_loss,
                    "learning_rate": current_lr,
                }
                pytorch_log_history.append(log_record)
                if global_step % config_ft.logging_steps == 0 or global_step == 1:
                    FINETUNE_LOGGER.info(
                        "PyTorch training loss | step=%s epoch=%s loss=%.4f lr=%.2e",
                        global_step,
                        epoch_index + 1,
                        last_loss,
                        current_lr,
                    )

    train_runtime = time.perf_counter() - train_start_time
    pytorch_train_result_metrics = {
        "train_loss": last_loss,
        "train_runtime": train_runtime,
        "global_step": global_step,
    }
    # Hugging Face checkpoint 형식으로 저장해 후속 로드와 Drive 복사가 가능하게 합니다.
    pytorch_model.save_pretrained(pytorch_model_dir)
    save_pretrained = getattr(pytorch_processor, "save_pretrained", None)
    if callable(save_pretrained):
        save_pretrained(pytorch_model_dir)
    pytorch_training_ran = True
    FINETUNE_LOGGER.info("Saved PyTorch fine-tuned model to %s", pytorch_model_dir)
else:
    FINETUNE_LOGGER.info("Skipped PyTorch full fine-tuning")


## PyTorch 평가와 caption 생성

**핵심:** 직접 학습한 model의 validation loss와 comparison/test caption을 계산하고, PyTorch 결과 폴더에 before/after report를 저장한다.


In [ ]:
# PyTorch 평가와 caption 생성

# 직접 PyTorch loop가 실행된 경우 validation loss를 계산하고 같은 comparison/test split에서 fine-tuned caption을 생성합니다.
# evaluate_full_finetuning_caption_run helper는 caption 생성, metric 계산, before/after JSON과 figure 저장을 한 번에 처리합니다.
pytorch_eval_metrics: dict[str, Any] | None = None
pytorch_comparison_metrics: dict[str, Any] | None = None
pytorch_outputs: dict[str, list[str]] | None = None

if pytorch_training_ran:
    pytorch_eval_metrics = evaluate_loss_pytorch(
        model=pytorch_model,
        dataloader=val_dataloader,
        device=device,
        torch_dtype=pytorch_model_torch_dtype,
    )

    release_cuda_memory(device)

    pytorch_run = evaluate_full_finetuning_caption_run(
        label="PyTorch fine-tuned",
        model=pytorch_model,
        processor=pytorch_processor,
        sample_splits=caption_splits,
        device=device,
        prompt=config_ft.prompt,
        max_new_tokens=config_ft.max_new_tokens,
        batch_size=config_ft.per_device_eval_batch_size,
        zero_shot_outputs=zero_shot_outputs,
        metric_samples=test_image_samples,
        comparison_samples=comparison_samples,
        output_dir=config_ft.pytorch_output_dir,
        model_dir=pytorch_model_dir,
        config_payload=pytorch_config_payload,
        train_result_metrics=pytorch_train_result_metrics,
        eval_metrics=pytorch_eval_metrics,
        log_history=pytorch_log_history,
        logger=FINETUNE_LOGGER,
    )
    pytorch_outputs = pytorch_run["outputs"]
    pytorch_comparison_metrics = pytorch_run["comparison_metrics"]
else:
    FINETUNE_LOGGER.info("Skipped PyTorch evaluation because PyTorch training did not run")


---

# Section 3: Hugging Face Trainer Full Fine-tuning

같은 split과 prompt를 사용하되 Hugging Face `Trainer`가 optimizer, scheduler, logging, eval loop를 관리하게 한다.


## Trainer용 새 base model 로드

**핵심:** PyTorch 학습으로 변경된 model을 해제하고, Trainer 비교는 같은 base model에서 새로 시작한다.


In [ ]:
# Trainer용 새 base model 로드

# Trainer 비교는 PyTorch로 이미 업데이트된 model을 재사용하지 않고 같은 base checkpoint를 새로 로드합니다.
# PyTorch model 참조와 CUDA cache를 먼저 정리해 같은 GPU에서 두 번째 full fine-tuning 경로를 실행할 메모리를 확보합니다.
trainer_processor = None
trainer_model = None
trainer_model_torch_dtype = None

if config_ft.run_trainer_comparison:
    if 'pytorch_model' in globals():
        del pytorch_model
    release_cuda_memory(device)

    trainer_bundle = load_model_and_processor(
        model_id=config_ft.model_name,
        device=device,
        torch_dtype=torch_dtype,
        prefer_flash_attention=False,
    )
    trainer_processor = trainer_bundle["processor"]
    trainer_model = trainer_bundle["model"]
    trainer_model_torch_dtype = trainer_bundle["torch_dtype"]
    FINETUNE_LOGGER.info("Loaded a fresh base model for Trainer with dtype=%s", trainer_model_torch_dtype)
else:
    FINETUNE_LOGGER.info("Skipped Trainer base model loading")


## Trainer 변수 준비

**핵심:** Trainer 방식의 최종 model 저장 경로와 report에 남길 실행 설정을 준비한다.


In [ ]:
# Trainer 변수 준비

# Trainer output directory, 최종 model directory, report payload, 결과 변수를 한곳에 준비합니다.
# Trainer 경로도 skip 가능하므로 후속 비교 셀이 None 값을 안전하게 받을 수 있게 초기화합니다.
trainer_work_dir = config_ft.trainer_output_dir / "trainer-output"
trainer_model_dir = config_ft.trainer_output_dir / "smolvlm-trainer-full-finetuned"
trainer_config_payload = build_config_payload(config_ft, extra={"sft_label_mask": "assistant_only_v1", "training_method": "trainer_v1"})
trainer = None
trainer_training_ran = False
trainer_train_result_metrics: dict[str, Any] | None = None
trainer_eval_metrics: dict[str, Any] | None = None
trainer_comparison_metrics: dict[str, Any] | None = None
trainer_outputs: dict[str, list[str]] | None = None
trainer_log_history: list[dict[str, Any]] = []

if config_ft.run_trainer_comparison and getattr(device, "type", None) == "cuda" and getattr(trainer_model.config, "use_cache", None) is not None:
    trainer_model.config.use_cache = False


## Trainer 구성

**핵심:** `TrainingArguments`, SFT collator, `Trainer`를 구성한다.


In [ ]:
# Trainer 구성

# TrainingArguments는 현재 transformers 버전에서 지원하는 인자만 공용 helper로 구성합니다.
# Trainer도 PyTorch loop와 같은 SFT collator를 사용해 assistant 답변 token만 loss 대상이 되게 맞춥니다.
if config_ft.run_trainer_comparison:
    # helper가 transformers 버전별 TrainingArguments 차이를 흡수해 지원되는 인자만 반환합니다.
    training_args_kwargs = build_training_arguments_kwargs(
        training_arguments_cls=transformers.TrainingArguments,
        output_dir=str(trainer_work_dir),
        config=config_ft,
        device=device,
        torch_dtype=trainer_model_torch_dtype,
    )
    # 중간 checkpoint 저장은 생략하고 최종 model만 별도로 저장합니다.
    training_args_kwargs["save_strategy"] = "no"
    training_args = transformers.TrainingArguments(**training_args_kwargs)

    # Trainer도 직접 loop와 같은 SFT collator를 사용해 label masking 조건을 맞춥니다.
    trainer_collate_fn = build_sft_collate_fn(
        trainer_processor,
        image_token_id=getattr(trainer_model.config, "image_token_id", None),
        prompt=config_ft.prompt,
    )
    # Trainer는 optimizer, scheduler, gradient accumulation, eval loop를 내부에서 관리합니다.
    trainer = transformers.Trainer(
        model=trainer_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=trainer_collate_fn,
        processing_class=trainer_processor,
    )
else:
    FINETUNE_LOGGER.info("Skipped Trainer setup")


## Trainer 학습

**핵심:** `trainer.train()`으로 full fine-tuning을 실행한 뒤 Trainer 방식의 최종 model과 processor를 저장한다.


In [ ]:
# Trainer 학습

# Trainer가 optimizer, scheduler, gradient accumulation, logging을 내부 orchestration으로 실행합니다.
# 최종 model과 processor를 별도 directory에 저장해 PyTorch loop 결과와 나란히 비교할 수 있게 합니다.
if config_ft.run_trainer_comparison:
    FINETUNE_LOGGER.info("Starting Trainer full fine-tuning")
    train_result = trainer.train()
    trainer_train_result_metrics = train_result.metrics
    trainer.save_model(trainer_model_dir)
    save_pretrained = getattr(trainer_processor, "save_pretrained", None)
    if callable(save_pretrained):
        save_pretrained(trainer_model_dir)
    trainer_training_ran = True
    FINETUNE_LOGGER.info("Saved Trainer fine-tuned model to %s", trainer_model_dir)
else:
    FINETUNE_LOGGER.info("Skipped Trainer full fine-tuning")


## Trainer 평가와 caption 생성

**핵심:** Trainer validation loss와 caption metric을 PyTorch loop와 같은 구조로 저장한다.


In [ ]:
# Trainer 평가와 caption 생성

# Trainer 학습이 완료된 경우 Trainer evaluate metric과 log_history를 수집합니다.
# caption 평가 단계는 PyTorch 경로와 같은 helper를 사용해 산출물 형식과 비교 조건을 동일하게 유지합니다.
if trainer_training_ran:
    FINETUNE_LOGGER.info("Starting Trainer evaluation")
    # Trainer 표준 evaluation metric을 먼저 수집합니다.
    trainer_eval_metrics = trainer.evaluate()
    # Trainer가 기록한 train/eval log를 report에 보존합니다.
    trainer_log_history = list(trainer.state.log_history)

    release_cuda_memory(device)

    # PyTorch 경로와 같은 helper로 caption/metric/report 형식을 통일합니다.
    trainer_run = evaluate_full_finetuning_caption_run(
        label="Trainer fine-tuned",
        model=trainer_model,
        processor=trainer_processor,
        sample_splits=caption_splits,
        device=device,
        prompt=config_ft.prompt,
        max_new_tokens=config_ft.max_new_tokens,
        batch_size=config_ft.per_device_eval_batch_size,
        zero_shot_outputs=zero_shot_outputs,
        metric_samples=test_image_samples,
        comparison_samples=comparison_samples,
        output_dir=config_ft.trainer_output_dir,
        model_dir=trainer_model_dir,
        config_payload=trainer_config_payload,
        train_result_metrics=trainer_train_result_metrics,
        eval_metrics=trainer_eval_metrics,
        log_history=trainer_log_history,
        logger=FINETUNE_LOGGER,
    )
    trainer_outputs = trainer_run["outputs"]
    trainer_comparison_metrics = trainer_run["comparison_metrics"]
else:
    FINETUNE_LOGGER.info("Skipped Trainer evaluation because Trainer training did not run")


---

# Section 4: Fine-tuning 방식 비교

Zero-shot, PyTorch loop fine-tuned, Trainer fine-tuned 결과를 같은 sample과 metric으로 비교한다.


## 비교 report 저장

**핵심:** 두 fine-tuning 방식의 test metric과 comparison caption을 하나의 JSON으로 묶어 저장한다.


In [ ]:
# 비교 report 저장

# zero-shot, PyTorch full FT, Trainer full FT 결과를 하나의 method_comparison.json으로 합칩니다.
# 2.5주차 LoRA fine-tuning 노트북은 이 파일을 읽어 full fine-tuning 기준 결과로 사용합니다.
method_comparison_report = build_full_finetuning_method_comparison_report(
    samples=comparison_samples,
    zero_shot_predictions=zero_shot_outputs["comparison"],
    pytorch_outputs=pytorch_outputs,
    trainer_outputs=trainer_outputs,
    pytorch_comparison_metrics=pytorch_comparison_metrics,
    trainer_comparison_metrics=trainer_comparison_metrics,
    pytorch_eval_metrics=pytorch_eval_metrics,
    trainer_eval_metrics=trainer_eval_metrics,
    pytorch_model_dir=pytorch_model_dir,
    trainer_model_dir=trainer_model_dir,
)

save_json(config_ft.output_dir / "method_comparison.json", method_comparison_report)
FINETUNE_LOGGER.info("Saved method comparison to %s", config_ft.output_dir / "method_comparison.json")


## 비교 시각화

**핵심:** 같은 comparison image에 대해 zero-shot, PyTorch fine-tuned, Trainer fine-tuned caption을 나란히 보고 test metric 막대그래프를 표시한다.


In [ ]:
# 비교 시각화

# method comparison report에서 caption 카드용 prediction set과 metric plot용 dict를 분리합니다.
# fine-tuned method가 하나도 평가되지 않은 경우 metric plot은 건너뛰고 로그만 남깁니다.
prediction_sets, metric_sets = build_full_finetuning_visualization_sets(method_comparison_report)

show_caption_comparison_cards(
    comparison_samples,
    prediction_sets=prediction_sets,
)

if metric_sets:
    show_caption_metric_comparison(
        metric_sets,
        title="Full fine-tuning method comparison",
    )
else:
    FINETUNE_LOGGER.info("No fine-tuned method was evaluated; skipped metric plot")


## 결과 Drive 복사

**핵심:** 결과 폴더가 클 수 있으므로 Drive 복사는 자동 실행하지 않는다. 필요할 때만 아래 주석 처리된 예시를 해제한다.


In [ ]:
# 결과 Drive 복사

# 학습 산출물을 Drive 프로젝트 폴더로 복사하는 선택형 마무리 셀입니다.
# model directory가 클 수 있으므로 실제 실행 환경에서 Drive 여유 공간과 복사 시간을 고려해야 합니다.
# Optional: full fine-tuned model directories can be large, so Drive copy is disabled by default.
# 필요할 때만 아래 예시의 주석을 해제하세요.
import shutil
drive_target_dir = Path(DRIVE_PROJECT) / config_ft.output_dir
shutil.copytree(config_ft.output_dir, drive_target_dir, dirs_exist_ok=True)
